In [1]:
import pandas as pd

files = {
    "temp": "Document_Corpus/dwd_temp/final_merged_dwd_temp.csv",
    "wind": "Document_Corpus/dwd_wind/final_merged_dwd_wind.csv",
    "solar": "Document_Corpus/dwd_solar/final_merged_dwd_solar.csv",
    "consumption": "Document_Corpus/power_consumption/final_merged_power_consumption.csv",
    "generation": "Document_Corpus/power_generation/final_merged_power_generation.csv",
    "capacity": "Document_Corpus/power_capacity/final_merged_power_capacity.csv",
    "price": "Document_Corpus/power_price/final_merged_power_price.csv"
}

for name, path in files.items():
    df = pd.read_csv(path, nrows=5)
    print(f"\n--- {name.upper()} ---")
    print(df.columns.tolist())
    print(df.head(2))


--- TEMP ---
['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU', 'RF_TU', 'eor']
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0         3987  1893010101     5  -12.3   84.0  eor
1         3987  1893010102     5  -10.9   84.0  eor

--- WIND ---
['STATIONS_ID', 'MESS_DATUM', 'QN_3', 'F', 'D', 'eor']
   STATIONS_ID  MESS_DATUM  QN_3    F    D  eor
0         3988  1893010100     5  5.4 -999  eor
1         3988  1893010101     5  5.5 -999  eor

--- SOLAR ---
['STATIONS_ID', 'MESS_DATUM', 'QN_592', 'ATMO_LBERG', 'FD_LBERG', 'FG_LBERG', 'SD_LBERG', 'ZENIT', 'MESS_DATUM_WOZ', 'eor']
   STATIONS_ID     MESS_DATUM  QN_592  ATMO_LBERG  FD_LBERG  FG_LBERG  \
0         3987  1945123101:10       1         NaN       0.0       0.0   
1         3987  1945123102:10       1         NaN       0.0       0.0   

   SD_LBERG   ZENIT MESS_DATUM_WOZ  eor  
0         0  146.08  1945123102:00  eor  
1         0  139.16  1945123103:00  eor  

--- CONSUMPTION ---
['Start date', 'End date', 'grid load [MWh] Calc

# DWD solar data

In [2]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/dwd_solar/final_merged_dwd_solar.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["MESS_DATUM"] = df["MESS_DATUM"].astype(str).str[:10]
df["datetime"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d%H")

# ---------------------------------------------------
# 3. REPLACE INVALID VALUES
# ---------------------------------------------------
df = df.replace([-999, -999.0, -9999, -9999.0], np.nan)

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "ATMO_LBERG": "atmospheric_radiation",
    "FD_LBERG": "diffuse_radiation",
    "FG_LBERG": "global_radiation",
    "SD_LBERG": "sunshine_duration",
    "ZENIT": "solar_zenith_angle"
})
df = df[[
    "datetime",
    "atmospheric_radiation",
    "diffuse_radiation",
    "global_radiation",
    "sunshine_duration",
    "solar_zenith_angle"
]]
# ---------------------------------------------------
# 5. OPTIONAL: CHECK RAW MISSINGNESS
# ---------------------------------------------------
print("Raw missing values:")
print(df[[
    "atmospheric_radiation",
    "diffuse_radiation",
    "global_radiation",
    "sunshine_duration",
    "solar_zenith_angle"
]].isna().sum())

# ---------------------------------------------------
# 6. AGGREGATE TO HOURLY NATIONAL LEVEL
# ---------------------------------------------------
agg_df = df.groupby("datetime").agg({
    "atmospheric_radiation": "mean",
    "diffuse_radiation": "mean",
    "global_radiation": "mean",
    "sunshine_duration": "mean",
    "solar_zenith_angle": "mean"
}).reset_index()

# ---------------------------------------------------
# 7. DOMAIN-AWARE IMPUTATION
# ---------------------------------------------------

# Fill only where missing AND no sunlight
mask = (
    (agg_df["sunshine_duration"] == 0) &
    (agg_df["atmospheric_radiation"].isna())
)
agg_df.loc[mask, "atmospheric_radiation"] = 0

mask = (
    (agg_df["sunshine_duration"] == 0) &
    (agg_df["diffuse_radiation"].isna())
)
agg_df.loc[mask, "diffuse_radiation"] = 0

mask = (
    (agg_df["sunshine_duration"] == 0) &
    (agg_df["global_radiation"].isna())
)
agg_df.loc[mask, "global_radiation"] = 0

# ---------------------------------------------------
# 8. SORT
# ---------------------------------------------------
agg_df = agg_df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 9. FINAL CHECK
# ---------------------------------------------------
print("\nAggregated missing values:")
print(agg_df.isna().sum())

print("\nSample rows:")
print(agg_df.sample(5))

# ---------------------------------------------------
# 10. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 11. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_dwd_solar_hourly.csv")
agg_df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

Raw missing values:
atmospheric_radiation    11656507
diffuse_radiation         2218498
global_radiation           386084
sunshine_duration          211223
solar_zenith_angle              0
dtype: int64

Aggregated missing values:
datetime                      0
atmospheric_radiation    166775
diffuse_radiation          2653
global_radiation           2653
sunshine_duration             0
solar_zenith_angle            0
dtype: int64

Sample rows:
                  datetime  atmospheric_radiation  diffuse_radiation  \
491825 2002-02-07 21:00:00             111.666667           0.000000   
577279 2011-11-08 11:00:00             118.818182          41.758621   
587424 2013-01-04 04:00:00             113.750000           0.000000   
395534 1991-02-13 18:00:00              87.800000           0.000000   
180417 1966-07-31 13:00:00                    NaN          95.500000   

        global_radiation  sunshine_duration  solar_zenith_angle  
491825          0.000000           0.000000        

# DWD WIND DATA

In [3]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/dwd_wind/final_merged_dwd_wind.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["MESS_DATUM"] = df["MESS_DATUM"].astype(str)
df["datetime"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d%H")

# ---------------------------------------------------
# 3. REPLACE INVALID PLACEHOLDERS
# ---------------------------------------------------
df = df.replace([-999, -999.0, -9999, -9999.0], np.nan)

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "F": "wind_speed",
    "D": "wind_direction"
})

# ---------------------------------------------------
# 5. KEEP ONLY RELEVANT COLUMNS
# ---------------------------------------------------
df = df[[
    "datetime",
    "wind_speed",
    "wind_direction"
]]

# ---------------------------------------------------
# 6. REMOVE IMPOSSIBLE WIND DIRECTION VALUES
# ---------------------------------------------------
df.loc[(df["wind_direction"] < 0) | (df["wind_direction"] > 360), "wind_direction"] = np.nan

# ---------------------------------------------------
# 7. CONVERT WIND DIRECTION TO CIRCULAR FEATURES
# ---------------------------------------------------
wind_dir_rad = np.deg2rad(df["wind_direction"])
df["wind_dir_sin"] = np.sin(wind_dir_rad)
df["wind_dir_cos"] = np.cos(wind_dir_rad)

# ---------------------------------------------------
# 8. AGGREGATE TO NATIONAL HOURLY LEVEL
# ---------------------------------------------------
agg_df = df.groupby("datetime").agg({
    "wind_speed": "mean",
    "wind_dir_sin": "mean",
    "wind_dir_cos": "mean",
    "wind_direction" : "mean"
}).reset_index()

# ---------------------------------------------------
# 9. SORT
# ---------------------------------------------------
agg_df = agg_df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 10. FINAL CHECK
# ---------------------------------------------------
print("Aggregated missing values:")
print(agg_df.isna().sum())

print("\nSample rows:")
print(agg_df.sample(5))

# ---------------------------------------------------
# 11. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 12. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_dwd_wind_hourly.csv")
agg_df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

Aggregated missing values:
datetime               0
wind_speed             0
wind_dir_sin      455130
wind_dir_cos      455130
wind_direction    455130
dtype: int64

Sample rows:
                  datetime  wind_speed  wind_dir_sin  wind_dir_cos  \
453703 1949-11-04 07:00:00    2.241667           NaN           NaN   
899581 2000-09-15 13:00:00    3.488664      0.538751     -0.423197   
867521 1997-01-18 17:00:00    2.448523     -0.212570     -0.281679   
914521 2002-05-31 01:00:00    1.702954     -0.226241     -0.209198   
986990 2010-09-05 14:00:00    3.723810      0.830611      0.319287   

        wind_direction  
453703             NaN  
899581      140.894309  
867521      184.599156  
914521      177.203390  
986990       76.483516  

Saved successfully to: Data_Corpus/semi_final\semi_final_dwd_wind_hourly.csv


# Temperature DATA

In [4]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/dwd_temp/final_merged_dwd_temp.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["MESS_DATUM"] = df["MESS_DATUM"].astype(str)
df["datetime"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d%H")

# ---------------------------------------------------
# 3. REPLACE INVALID VALUES
# ---------------------------------------------------
df = df.replace([-999, -999.0, -9999, -9999.0], np.nan)

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "TT_TU": "air_temperature",
    "RF_TU": "relative_humidity"
})

# ---------------------------------------------------
# 5. KEEP ONLY RELEVANT COLUMNS
# ---------------------------------------------------
df = df[[
    "datetime",
    "air_temperature",
    "relative_humidity"
]]

# ---------------------------------------------------
# 6. CLEAN IMPOSSIBLE HUMIDITY VALUES
# ---------------------------------------------------
df.loc[(df["relative_humidity"] < 0) | (df["relative_humidity"] > 100), "relative_humidity"] = np.nan

# ---------------------------------------------------
# 7. AGGREGATE TO NATIONAL HOURLY LEVEL
# ---------------------------------------------------
agg_df = df.groupby("datetime").agg({
    "air_temperature": "mean",
    "relative_humidity": "mean"
}).reset_index()

# ---------------------------------------------------
# 8. SORT
# ---------------------------------------------------
agg_df = agg_df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 9. FINAL CHECK
# ---------------------------------------------------
print("Aggregated missing values:")
print(agg_df.isna().sum())

print("\nSample rows:")
print(agg_df.sample(5))

# ---------------------------------------------------
# 10. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 11. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_dwd_temp_hourly.csv")
agg_df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

Aggregated missing values:
datetime             0
air_temperature      0
relative_humidity    0
dtype: int64

Sample rows:
                   datetime  air_temperature  relative_humidity
546723  1955-05-17 16:00:00        13.323711          76.216495
684491  1971-02-03 00:00:00        -4.035714          77.693694
765227  1980-04-20 00:00:00         0.806195          81.964912
931835  1999-04-23 00:00:00         7.229268          88.266667
1008854 2008-02-04 03:00:00        -0.424025          78.455852

Saved successfully to: Data_Corpus/semi_final\semi_final_dwd_temp_hourly.csv


# Power Consumption

In [5]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv(
    "Document_Corpus/power_consumption/final_merged_power_consumption.csv"
)

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["datetime"] = pd.to_datetime(df["Start date"])

# Floor to strict hourly timestamps
df["datetime"] = df["datetime"].dt.floor("h")

# ---------------------------------------------------
# 3. KEEP REQUIRED COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "grid load [MWh] Calculated resolutions":
        "grid_load_mwh",

    "Hydro pumped storage [MWh] Calculated resolutions":
        "hydro_pumped_storage_mwh",

    "Residual load [MWh] Calculated resolutions":
        "residual_load_mwh"
})

df = df[
    [
        "datetime",
        "grid_load_mwh",
        "hydro_pumped_storage_mwh",
        "residual_load_mwh"
    ]
]

# ---------------------------------------------------
# 4. CLEAN NUMERIC COLUMNS
# ---------------------------------------------------
for col in df.columns:
    if col != "datetime":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )

        df[col] = df[col].replace("-", np.nan)

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ---------------------------------------------------
# 5. AGGREGATE TO STRICT HOURLY LEVEL
# IMPORTANT:
# Use MEAN instead of SUM because duplicate
# timestamps are DST (clock-change) hours.
# Summing would artificially double demand.
# ---------------------------------------------------
consumption_df = (
    df.groupby("datetime", as_index=False)
      .mean(numeric_only=True)
)

# ---------------------------------------------------
# 6. SORT
# ---------------------------------------------------
consumption_df = (
    consumption_df
    .sort_values("datetime")
    .reset_index(drop=True)
)

# ---------------------------------------------------
# 7. FINAL CHECKS
# ---------------------------------------------------
print("Final shape:", consumption_df.shape)

print("\nMissing values:")
print(consumption_df.isna().sum())

print(
    "\nDuplicate datetimes:",
    consumption_df["datetime"].duplicated().sum()
)

print("\nSample rows:")
print(consumption_df.sample(10))

print("\nDescribe:")
print(consumption_df.describe())

# ---------------------------------------------------
# 8. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 9. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(
    output_dir,
    "semi_final_power_consumption_hourly.csv"
)

consumption_df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\893246833.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(df["Start date"])


Final shape: (98580, 4)

Missing values:
datetime                     0
grid_load_mwh                0
hydro_pumped_storage_mwh    96
residual_load_mwh            0
dtype: int64

Duplicate datetimes: 0

Sample rows:
                 datetime  grid_load_mwh  hydro_pumped_storage_mwh  \
64357 2022-05-05 21:00:00       57918.75                     69.75   
48373 2020-07-08 19:00:00       59730.50                   2396.50   
77223 2023-10-24 00:00:00       46036.00                   3322.25   
36407 2019-02-26 03:00:00       49493.75                   4144.50   
11267 2016-04-14 13:00:00       68638.98                    281.23   
42990 2019-11-27 11:00:00       75869.00                    281.25   
73750 2023-06-01 07:00:00       56145.00                     10.00   
57819 2021-08-06 10:00:00       63126.00                    214.25   
69832 2022-12-20 00:00:00       50682.00                   6211.50   
80043 2024-02-18 12:00:00       54493.00                   4884.50   

       residu

# Power Generation

In [6]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/power_generation/final_merged_power_generation.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["datetime"] = pd.to_datetime(df["Start date"])

# Floor to strict hourly timestamps
df["datetime"] = df["datetime"].dt.floor("h")

# ---------------------------------------------------
# 3. DROP UNUSED COLUMNS
# ---------------------------------------------------
drop_cols = ["Start date", "End date", "source_file"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "Biomass [MWh] Calculated resolutions": "generation_biomass_mwh",
    "Hydropower [MWh] Calculated resolutions": "generation_hydropower_mwh",
    "Wind offshore [MWh] Calculated resolutions": "generation_wind_offshore_mwh",
    "Wind onshore [MWh] Calculated resolutions": "generation_wind_onshore_mwh",
    "Photovoltaics [MWh] Calculated resolutions": "generation_photovoltaics_mwh",
    "Other renewable [MWh] Calculated resolutions": "generation_other_renewable_mwh",
    "Nuclear [MWh] Calculated resolutions": "generation_nuclear_mwh",
    "Lignite [MWh] Calculated resolutions": "generation_lignite_mwh",
    "Hard coal [MWh] Calculated resolutions": "generation_hard_coal_mwh",
    "Fossil gas [MWh] Calculated resolutions": "generation_fossil_gas_mwh",
    "Hydro pumped storage [MWh] Calculated resolutions": "generation_hydro_pumped_storage_mwh",
    "Other conventional [MWh] Calculated resolutions": "generation_other_conventional_mwh"
})

# ---------------------------------------------------
# 5. CLEAN NUMERIC COLUMNS
# ---------------------------------------------------
for col in df.columns:
    if col != "datetime":
        df[col] = df[col].astype(str).str.replace(",", "", regex=False)
        df[col] = df[col].replace("-", np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------
# 6. AGGREGATE TO STRICT HOURLY LEVEL
# IMPORTANT: use SUM because values are in MWh
# ---------------------------------------------------
agg_df = df.groupby("datetime", as_index=False).mean(numeric_only=True)

# ---------------------------------------------------
# 7. SORT
# ---------------------------------------------------
agg_df = agg_df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 8. FINAL CHECKS
# ---------------------------------------------------
print("Final shape:", agg_df.shape)

print("\nFinal missing values:")
print(agg_df.isna().sum())

print("\nDuplicate datetimes:", agg_df["datetime"].duplicated().sum())

print("\nSample rows:")
print(agg_df.sample(10))

print("\nDescribe:")
print(agg_df.describe())

# ---------------------------------------------------
# 9. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 10. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_power_generation_hourly.csv")
agg_df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\586354298.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Document_Corpus/power_generation/final_merged_power_generation.csv")
C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\586354298.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(df["Start date"])


Final shape: (98580, 13)

Final missing values:
datetime                                   0
generation_biomass_mwh                     0
generation_hydropower_mwh                  0
generation_wind_offshore_mwh               0
generation_wind_onshore_mwh                0
generation_photovoltaics_mwh               0
generation_other_renewable_mwh            23
generation_nuclear_mwh                 18993
generation_lignite_mwh                     0
generation_hard_coal_mwh                   0
generation_fossil_gas_mwh                  0
generation_hydro_pumped_storage_mwh        0
generation_other_conventional_mwh          0
dtype: int64

Duplicate datetimes: 0

Sample rows:
                 datetime  generation_biomass_mwh  generation_hydropower_mwh  \
52441 2020-12-25 07:00:00                 4563.25                    1436.75   
26886 2018-01-25 09:00:00                 4825.25                    1811.00   
67895 2022-09-30 07:00:00                 4690.75                    2308.50

# Power Capacity

In [7]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/power_capacity/final_merged_power_capacity.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["datetime"] = pd.to_datetime(df["Start date"])

# ---------------------------------------------------
# 3. DROP UNUSED COLUMNS
# ---------------------------------------------------
drop_cols = ["Start date", "End date", "source_file"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "Biomass [MW] Calculated resolutions": "capacity_biomass_mw",
    "Hydropower [MW] Calculated resolutions": "capacity_hydropower_mw",
    "Wind offshore [MW] Calculated resolutions": "capacity_wind_offshore_mw",
    "Wind onshore [MW] Calculated resolutions": "capacity_wind_onshore_mw",
    "Photovoltaics [MW] Calculated resolutions": "capacity_photovoltaics_mw",
    "Other renewable [MW] Calculated resolutions": "capacity_other_renewable_mw",
    "Nuclear [MW] Calculated resolutions": "capacity_nuclear_mw",
    "Lignite [MW] Calculated resolutions": "capacity_lignite_mw",
    "Hard coal [MW] Calculated resolutions": "capacity_hard_coal_mw",
    "Fossil gas [MW] Calculated resolutions": "capacity_fossil_gas_mw",
    "Hydro pumped storage [MW] Calculated resolutions": "capacity_hydro_pumped_storage_mw",
    "Other conventional [MW] Calculated resolutions": "capacity_other_conventional_mw"
})

# ---------------------------------------------------
# 5. CLEAN NUMERIC COLUMNS
# ---------------------------------------------------
for col in df.columns:
    if col != "datetime":
        df[col] = df[col].astype(str).str.replace(",", "", regex=False)
        df[col] = df[col].replace("-", np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------
# 6. REMOVE DST DUPLICATES
# (clock rollback in late October causes duplicate 02:00 rows)
# ---------------------------------------------------
df = df.drop_duplicates(subset=["datetime"])

# ---------------------------------------------------
# 7. DROP FULLY EMPTY / USELESS COLUMNS
# ---------------------------------------------------
if "capacity_nuclear_mw" in df.columns:
    if df["capacity_nuclear_mw"].isna().all():
        df = df.drop(columns=["capacity_nuclear_mw"])

# ---------------------------------------------------
# 8. SORT
# ---------------------------------------------------
df = df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 9. FINAL CHECK
# ---------------------------------------------------
print("Final shape:", df.shape)

print("\nFinal missing values:")
print(df.isna().sum())

print("\nDuplicate datetimes:", df["datetime"].duplicated().sum())

print("\nSample rows:")
print(df.sample(5))

# ---------------------------------------------------
# 10. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 11. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_power_capacity_hourly.csv")
df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\1783026529.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(df["Start date"])
C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\1783026529.py:45: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace("-", np.nan)


Final shape: (98580, 12)

Final missing values:
capacity_biomass_mw                     0
capacity_hydropower_mw              61241
capacity_wind_offshore_mw           78879
capacity_wind_onshore_mw                0
capacity_photovoltaics_mw               0
capacity_other_renewable_mw         69976
capacity_lignite_mw                 78879
capacity_hard_coal_mw               69976
capacity_fossil_gas_mw                  0
capacity_hydro_pumped_storage_mw    61241
capacity_other_conventional_mw      78879
datetime                                0
dtype: int64

Duplicate datetimes: 0

Sample rows:
       capacity_biomass_mw  capacity_hydropower_mw  capacity_wind_offshore_mw  \
45537               7833.0                     NaN                        NaN   
32190               7073.0                     NaN                        NaN   
23475               7080.0                     NaN                        NaN   
376                 6808.0                     NaN                       

# Power Price

In [8]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD
# ---------------------------------------------------
df = pd.read_csv("Document_Corpus/power_price/final_merged_power_price.csv")

# ---------------------------------------------------
# 2. CLEAN DATETIME
# ---------------------------------------------------
df["datetime"] = pd.to_datetime(df["Start date"])

# ---------------------------------------------------
# 3. KEEP ONLY RELEVANT GERMAN PRICE COLUMNS
# ---------------------------------------------------
df = df[[
    "datetime",
    "Germany/Luxembourg [€/MWh] Calculated resolutions",
    "DE/AT/LU [€/MWh] Calculated resolutions"
]]

# ---------------------------------------------------
# 4. RENAME COLUMNS
# ---------------------------------------------------
df = df.rename(columns={
    "Germany/Luxembourg [€/MWh] Calculated resolutions": "price_germany_luxembourg",
    "DE/AT/LU [€/MWh] Calculated resolutions": "price_de_at_lu"
})

# ---------------------------------------------------
# 5. CLEAN NUMERIC COLUMNS
# ---------------------------------------------------
for col in ["price_germany_luxembourg", "price_de_at_lu"]:
    df[col] = df[col].astype(str).replace("-", np.nan)
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------
# 6. CREATE UNIFIED GERMAN PRICE SERIES
# Use Germany/Luxembourg first, fallback to DE/AT/LU
# ---------------------------------------------------
df["price_eur_per_mwh"] = df["price_germany_luxembourg"].fillna(df["price_de_at_lu"])

# ---------------------------------------------------
# 7. KEEP ONLY FINAL PRICE COLUMN
# ---------------------------------------------------
df = df[[
    "datetime",
    "price_eur_per_mwh"
]]

# ---------------------------------------------------
# 8. REMOVE DST DUPLICATES
# ---------------------------------------------------
df = df.drop_duplicates(subset=["datetime"])

# ---------------------------------------------------
# 9. SORT
# ---------------------------------------------------
df = df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 10. FINAL CHECK
# ---------------------------------------------------
print("Final shape:", df.shape)

print("\nFinal missing values:")
print(df.isna().sum())

print("\nDuplicate datetimes:", df["datetime"].duplicated().sum())

print("\nSample rows:")
print(df.sample(5))

print("\nDescribe:")
print(df.describe())

# ---------------------------------------------------
# 11. CREATE OUTPUT DIRECTORY
# ---------------------------------------------------
output_dir = "Data_Corpus/semi_final"
os.makedirs(output_dir, exist_ok=True)

# ---------------------------------------------------
# 12. SAVE CSV
# ---------------------------------------------------
output_path = os.path.join(output_dir, "semi_final_power_price_hourly.csv")
df.to_csv(output_path, index=False)

print(f"\nSaved successfully to: {output_path}")

C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\2417826143.py:8: DtypeWarning: Columns (2,3,4,7,8,10,11,13,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("Document_Corpus/power_price/final_merged_power_price.csv")
C:\Users\Kamal_FOM\AppData\Local\Temp\ipykernel_17756\2417826143.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["datetime"] = pd.to_datetime(df["Start date"])


Final shape: (98580, 2)

Final missing values:
datetime              0
price_eur_per_mwh    96
dtype: int64

Duplicate datetimes: 0

Sample rows:
                 datetime  price_eur_per_mwh
58373 2021-08-29 12:00:00              82.84
55414 2021-04-28 05:00:00              55.70
54345 2021-03-14 15:00:00               0.13
5202  2015-08-05 19:00:00              46.90
81772 2024-04-30 14:00:00              10.00

Describe:
                            datetime  price_eur_per_mwh
count                          98580       98484.000000
mean   2020-08-15 23:22:43.311016704          73.658061
min              2015-01-01 00:00:00        -500.000000
25%              2017-10-23 23:45:00          30.200000
50%              2020-08-15 23:30:00          46.860000
75%              2023-06-08 23:15:00          92.542500
max              2026-03-31 23:00:00         936.280000
std                              NaN          80.284497

Saved successfully to: Data_Corpus/semi_final\semi_final_power_price

# Final Integration

In [9]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------
# 1. LOAD SEMI-FINAL DATASETS
# ---------------------------------------------------
solar = pd.read_csv("Data_Corpus/semi_final/semi_final_dwd_solar_hourly.csv")
wind = pd.read_csv("Data_Corpus/semi_final/semi_final_dwd_wind_hourly.csv")
temp = pd.read_csv("Data_Corpus/semi_final/semi_final_dwd_temp_hourly.csv")
gen = pd.read_csv("Data_Corpus/semi_final/semi_final_power_generation_hourly.csv")
cap = pd.read_csv("Data_Corpus/semi_final/semi_final_power_capacity_hourly.csv")
price = pd.read_csv("Data_Corpus/semi_final/semi_final_power_price_hourly.csv")
cons = pd.read_csv("Data_Corpus/semi_final/semi_final_power_consumption_hourly.csv")

# ---------------------------------------------------
# 2. CONVERT DATETIME
# ---------------------------------------------------
datasets = [solar, wind, temp, cons, gen, cap, price]
for df in datasets:
    df["datetime"] = pd.to_datetime(df["datetime"])

# ---------------------------------------------------
# 3. MERGE ALL ON GENERATION (BASE DATASET)
# ---------------------------------------------------
master_df = gen.copy()
master_df = master_df.merge(cons, on="datetime", how="left")
master_df = master_df.merge(solar, on="datetime", how="left")
master_df = master_df.merge(wind, on="datetime", how="left")
master_df = master_df.merge(temp, on="datetime", how="left")
master_df = master_df.merge(cap, on="datetime", how="left")
master_df = master_df.merge(price, on="datetime", how="left")

# ---------------------------------------------------
# 4. SORT
# ---------------------------------------------------
master_df = master_df.sort_values("datetime").reset_index(drop=True)

# ---------------------------------------------------
# 5. FILL OFFSHORE WIND CAPACITY (2015–2023 ONLY)
# Keep actual dataset values for 2024+
# ---------------------------------------------------
master_df["year"] = master_df["datetime"].dt.year

offshore_capacity_map = {
    2015: 3300,
    2016: 4100,
    2017: 5400,
    2018: 6600,
    2019: 7700,
    2020: 7800,
    2021: 7900,
    2022: 8100,
    2023: 8400
}

if "capacity_wind_offshore_mw" in master_df.columns:
    mask = (
        master_df["capacity_wind_offshore_mw"].isna() &
        master_df["year"].isin(offshore_capacity_map.keys())
    )
    master_df.loc[mask, "capacity_wind_offshore_mw"] = master_df.loc[mask, "year"].map(offshore_capacity_map)

# ---------------------------------------------------
# 6. SAVE FULL MASTER DATASET
# ---------------------------------------------------
output_dir = "Data_Corpus/final"
os.makedirs(output_dir, exist_ok=True)

full_output_path = os.path.join(output_dir, "master_energy_dataset_full.csv")
master_df.to_csv(full_output_path, index=False)

print("FULL MASTER DATASET SAVED")
print("Path:", full_output_path)
print("Shape:", master_df.shape)

print("\nTop missing values in FULL dataset:")
print(master_df.isna().sum().sort_values(ascending=False).head(20))

# ---------------------------------------------------
# 7. CREATE MODELING DATASET
# ---------------------------------------------------
model_df = master_df.copy()

# ---------------------------------------------------
# 8. DROP HIGH-MISSING / LOW-PRIORITY CAPACITY COLUMNS
# ---------------------------------------------------
drop_cols = [
    "capacity_other_conventional_mw",
    "capacity_lignite_mw",
    "capacity_hard_coal_mw",
    "capacity_other_renewable_mw",
    "capacity_hydropower_mw",
    "capacity_hydro_pumped_storage_mw"
]

model_df = model_df.drop(columns=[col for col in drop_cols if col in model_df.columns])

# ---------------------------------------------------
# 9. INTERPOLATE SMALL GAPS IN FEATURES ONLY
# (Do NOT interpolate generation targets)
# ---------------------------------------------------
feature_cols_to_interpolate = [
    "atmospheric_radiation",
    "diffuse_radiation",
    "global_radiation",
    "sunshine_duration",
    "solar_zenith_angle",
    "wind_speed",
    "wind_direction",
    "wind_dir_sin",
    "wind_dir_cos",
    "air_temperature",
    "relative_humidity",
    "price_eur_per_mwh",
    "hydro_pumped_storage_mwh"
]

for col in feature_cols_to_interpolate:
    if col in model_df.columns:
        model_df[col] = model_df[col].interpolate(method="linear", limit_direction="both")

print("\nTop missing values in FULL dataset:")
print(model_df.isna().sum().sort_values(ascending=False).head(20))

print(model_df.info())

FULL MASTER DATASET SAVED
Path: Data_Corpus/final\master_energy_dataset_full.csv
Shape: (98580, 40)

Top missing values in FULL dataset:
capacity_other_conventional_mw      78879
capacity_lignite_mw                 78879
capacity_other_renewable_mw         69976
capacity_hard_coal_mw               69976
capacity_hydropower_mw              61241
capacity_hydro_pumped_storage_mw    61241
generation_nuclear_mwh              18993
sunshine_duration                     743
atmospheric_radiation                 743
solar_zenith_angle                    743
diffuse_radiation                     743
global_radiation                      743
hydro_pumped_storage_mwh               96
price_eur_per_mwh                      96
generation_other_renewable_mwh         23
datetime                                0
generation_wind_onshore_mwh             0
generation_photovoltaics_mwh            0
generation_hydropower_mwh               0
generation_wind_offshore_mwh            0
dtype: int64

Top missi

In [10]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98580 entries, 0 to 98579
Data columns (total 34 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   datetime                             98580 non-null  datetime64[ns]
 1   generation_biomass_mwh               98580 non-null  float64       
 2   generation_hydropower_mwh            98580 non-null  float64       
 3   generation_wind_offshore_mwh         98580 non-null  float64       
 4   generation_wind_onshore_mwh          98580 non-null  float64       
 5   generation_photovoltaics_mwh         98580 non-null  float64       
 6   generation_other_renewable_mwh       98557 non-null  float64       
 7   generation_nuclear_mwh               79587 non-null  float64       
 8   generation_lignite_mwh               98580 non-null  float64       
 9   generation_hard_coal_mwh             98580 non-null  float64       
 10  generation

In [11]:
model_df.columns.tolist()

['datetime',
 'generation_biomass_mwh',
 'generation_hydropower_mwh',
 'generation_wind_offshore_mwh',
 'generation_wind_onshore_mwh',
 'generation_photovoltaics_mwh',
 'generation_other_renewable_mwh',
 'generation_nuclear_mwh',
 'generation_lignite_mwh',
 'generation_hard_coal_mwh',
 'generation_fossil_gas_mwh',
 'generation_hydro_pumped_storage_mwh',
 'generation_other_conventional_mwh',
 'grid_load_mwh',
 'hydro_pumped_storage_mwh',
 'residual_load_mwh',
 'atmospheric_radiation',
 'diffuse_radiation',
 'global_radiation',
 'sunshine_duration',
 'solar_zenith_angle',
 'wind_speed',
 'wind_dir_sin',
 'wind_dir_cos',
 'wind_direction',
 'air_temperature',
 'relative_humidity',
 'capacity_biomass_mw',
 'capacity_wind_offshore_mw',
 'capacity_wind_onshore_mw',
 'capacity_photovoltaics_mw',
 'capacity_fossil_gas_mw',
 'price_eur_per_mwh',
 'year']

In [12]:
# ---------------------------------------------------
# 10. OPTIONAL: KEEP ONLY CORE MODELING COLUMNS
# (recommended for first modeling version)
# ---------------------------------------------------
core_cols = [
    "datetime",

    # TARGETS
    "generation_photovoltaics_mwh",
    "generation_wind_onshore_mwh",
    "generation_wind_offshore_mwh",

    # WEATHER
    "atmospheric_radiation",
    "diffuse_radiation",
    "global_radiation",
    "sunshine_duration",
    "solar_zenith_angle",
    "wind_speed",
    "wind_direction",
    "wind_dir_sin",
    "wind_dir_cos",
    "air_temperature",
    "relative_humidity",

    # CAPACITY
    "capacity_biomass_mw",
    "capacity_wind_onshore_mw",
    "capacity_wind_offshore_mw",
    "capacity_photovoltaics_mw",
    "capacity_fossil_gas_mw",

    # DEMAND
    "grid_load_mwh",
    "hydro_pumped_storage_mwh",

    # MARKET
    "price_eur_per_mwh"
]

core_cols = [col for col in core_cols if col in model_df.columns]
model_df = model_df[core_cols]

# ---------------------------------------------------
# 11. FINAL CHECK
# ---------------------------------------------------
print("\nMODELING DATASET SHAPE:", model_df.shape)

print("\nMissing values in MODELING dataset:")
print(model_df.isna().sum().sort_values(ascending=False))

print("\nUnique offshore capacity values by year:")
temp_check = master_df.groupby("year")["capacity_wind_offshore_mw"].unique()
print(temp_check)

print("\nSample rows from MODELING dataset:")
print(model_df.sample(5))

# ---------------------------------------------------
# 12. DROP HELPER YEAR COLUMN FROM FULL DATASET IF NEEDED
# ---------------------------------------------------
if "year" in master_df.columns:
    master_df = master_df.drop(columns=["year"])

# ---------------------------------------------------
# 13. SAVE MODELING DATASET
# ---------------------------------------------------
model_output_path = os.path.join(output_dir, "master_energy_dataset_modeling.csv")
model_df.to_csv(model_output_path, index=False)

print("\nMODELING DATASET SAVED")
print("Path:", model_output_path)


MODELING DATASET SHAPE: (98580, 23)

Missing values in MODELING dataset:
datetime                        0
generation_photovoltaics_mwh    0
generation_wind_onshore_mwh     0
generation_wind_offshore_mwh    0
atmospheric_radiation           0
diffuse_radiation               0
global_radiation                0
sunshine_duration               0
solar_zenith_angle              0
wind_speed                      0
wind_direction                  0
wind_dir_sin                    0
wind_dir_cos                    0
air_temperature                 0
relative_humidity               0
capacity_biomass_mw             0
capacity_wind_onshore_mw        0
capacity_wind_offshore_mw       0
capacity_photovoltaics_mw       0
capacity_fossil_gas_mw          0
grid_load_mwh                   0
hydro_pumped_storage_mwh        0
price_eur_per_mwh               0
dtype: int64

Unique offshore capacity values by year:
year
2015    [3300.0]
2016    [4100.0]
2017    [5400.0]
2018    [6600.0]
2019    [7700.0]